# Agri RAG — v15: 10-fold CE Ensemble + Pretrained Reranker Ensemble

**v13 = 0.96835 LB** (5-fold CE ensemble + triple BM25). Target: 0.98+

**Error analysis findings:**
- Recall is PERFECT (100% rel3 docs in top-100). More retrievers/HyDE won't help.
- CE ranking is the bottleneck: loses 0.048 nDCG vs oracle.
- Failures are needle-in-haystack: 1 rel3 doc among 99 rel0 on small topics.

**This notebook's improvements:**
1. **10-fold CE ensemble** (was 5-fold) — more diversity, each CE sees 90% of topics
2. **Pretrained reranker** (BAAI/bge-reranker-v2-m3) — zero-shot, independent signal, no API needed
3. **Weighted ensemble** of finetuned CE + pretrained reranker, α tuned on val
4. Top-5 from ensembled scores → submission

In [ ]:
# Cell 1: Setup
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'torch', 'torchvision', 'torchaudio'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
                       'torch==2.10.0', 'torchvision', 'torchaudio',
                       '--index-url', 'https://download.pytorch.org/whl/cu126'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rank_bm25', 'nltk'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.environ['TORCH_COMPILE_DISABLE'] = '1'
import torch, torch._dynamo
torch._dynamo.config.suppress_errors = True

print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    device = "cpu"
print(f"Device: {device}")

In [ ]:
# Cell 2: Load data
import os, json, glob, re
import numpy as np
import pandas as pd

doc_path = glob.glob('/kaggle/input/**/documents.csv', recursive=True)
DATA = os.path.dirname(doc_path[0])

MODELS = None
for candidate in ['/kaggle/input/agri-rag-models',
                  '/kaggle/input/datasets/hamzattiamiyuustaz/agri-rag-models']:
    if os.path.isdir(candidate):
        MODELS = candidate
        break
if MODELS is None:
    model_hits = glob.glob('/kaggle/input/**/hnm_final_r2', recursive=True)
    if model_hits:
        MODELS = os.path.dirname(model_hits[0])
    else:
        print("Available:", os.listdir('/kaggle/input'))
        raise FileNotFoundError("Cannot find models directory")

print(f"Data: {DATA}\nModels: {MODELS}")

docs = pd.read_csv(f"{DATA}/documents.csv", dtype={"document_id": str})
train_q = pd.read_csv(f"{DATA}/train_queries.csv", dtype={"query_id": str})
test_q = pd.read_csv(f"{DATA}/test_queries.csv", dtype={"query_id": str})
qrels = pd.read_csv(f"{DATA}/qrels_train.csv", dtype={"query_id": str, "document_id": str})

doc_ids = docs["document_id"].tolist()
corpus = (docs["title"].fillna("") + ". " + docs["text"].fillna("")).tolist()
doc_map = dict(zip(doc_ids, corpus))
q_map = dict(zip(train_q["query_id"], train_q["query"]))
tq_map = dict(zip(test_q["query_id"], test_q["query"]))

def ndcg_at_k(ranked, gains, k=5):
    dcg = sum((2**gains.get(d, 0) - 1) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(gains.values(), reverse=True)[:k]
    idcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

qrels_g = {qid: dict(zip(g["document_id"], g["relevance"])) for qid, g in qrels.groupby("query_id")}
print(f"docs {len(docs)} | train {len(train_q)} | test {len(test_q)} | qrels {len(qrels)}")

In [ ]:
# Cell 3: Topic split + 10-fold for ensemble
from collections import defaultdict

pos_docs_per_q = {}
for qid, gains in qrels_g.items():
    pos = frozenset(d for d, r in gains.items() if r >= 2)
    pos_docs_per_q[qid] = pos

qid_to_topic, topic_id, doc_to_topic = {}, 0, {}
for qid, pos in pos_docs_per_q.items():
    existing = set(doc_to_topic[d] for d in pos if d in doc_to_topic)
    if existing:
        merge_to = min(existing)
        qid_to_topic[qid] = merge_to
        for d in pos: doc_to_topic[d] = merge_to
        for tid in existing - {merge_to}:
            for q, t in qid_to_topic.items():
                if t == tid: qid_to_topic[q] = merge_to
            for d, t in doc_to_topic.items():
                if t == tid: doc_to_topic[d] = merge_to
    else:
        qid_to_topic[qid] = topic_id
        for d in pos: doc_to_topic[d] = topic_id
        topic_id += 1

topics = defaultdict(list)
for qid, tid in qid_to_topic.items():
    topics[tid].append(qid)

# 80/20 topic split for val (same seed as v13 for comparability)
rng = np.random.RandomState(42)
tids = sorted(topics.keys()); rng.shuffle(tids)
n_val = max(1, len(tids) // 5)
val_qids = set()
for tid in tids[:n_val]: val_qids.update(topics[tid])
fit_qids = set()
for tid in tids[n_val:]: fit_qids.update(topics[tid])

# 10-fold split of ALL topics for final ensemble
N_FOLDS = 10
rng3 = np.random.RandomState(456)
all_tids = sorted(topics.keys()); rng3.shuffle(all_tids)
all_folds = []
fold_size = len(all_tids) // N_FOLDS
for i in range(N_FOLDS):
    s = i * fold_size
    e = s + fold_size if i < N_FOLDS - 1 else len(all_tids)
    fq = set()
    for tid in all_tids[s:e]: fq.update(topics[tid])
    all_folds.append(fq)

print(f"Topics: {len(topics)} | FIT: {len(fit_qids)} | VAL: {len(val_qids)}")
print(f"{N_FOLDS}-fold all: {[len(f) for f in all_folds]}")

In [ ]:
# Cell 4: Candidates — triple BM25 (proven best, v11/v13)
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

bienc = SentenceTransformer(os.path.join(MODELS, "hnm_final_r2"), device=device)
emb = bienc.encode(["passage: " + c for c in corpus],
                   normalize_embeddings=True, batch_size=64, show_progress_bar=True)

def dense_topk(query, k=100):
    qe = bienc.encode(["query: " + query], normalize_embeddings=True)
    return [doc_ids[i] for i in np.argsort(-(emb @ qe.T).ravel())[:k]]

tokenized_basic = [doc.lower().split() for doc in corpus]
bm25_basic = BM25Okapi(tokenized_basic)

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
def stem_tok(text):
    tokens = re.findall(r'\w+', text.lower())
    return [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 1]
tokenized_stemmed = [stem_tok(doc) for doc in corpus]
bm25_stemmed = BM25Okapi(tokenized_stemmed)

def rrf_fuse(lists, k=60):
    scores = {}
    for cands in lists:
        for pos, d in enumerate(cands):
            scores[d] = scores.get(d, 0) + 1.0 / (k + pos + 1)
    return sorted(scores.keys(), key=lambda d: -scores[d])

def triple_bm25(query, k=100):
    dense = dense_topk(query, k)
    bm25_b = [doc_ids[i] for i in np.argsort(-bm25_basic.get_scores(query.lower().split()))[:k]]
    bm25_s = [doc_ids[i] for i in np.argsort(-bm25_stemmed.get_scores(stem_tok(query)))[:k]]
    return rrf_fuse([dense, bm25_b, bm25_s])[:k]

print("Train candidates (dense)...")
train_cands = {qid: dense_topk(q_map[qid], 100) for qid in train_q["query_id"]}

print("Val candidates (triple BM25)...")
val_cands = {qid: triple_bm25(q_map[qid], 100) for qid in val_qids if qid in q_map}

print("Test candidates (triple BM25)...")
test_cands = {qid: triple_bm25(q, 100) for qid, q in zip(test_q["query_id"], test_q["query"])}

del bienc, emb; torch.cuda.empty_cache()
print(f"Train: {len(train_cands)} | Val: {len(val_cands)} | Test: {len(test_cands)}")

In [ ]:
# Cell 5: CE helpers
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

def build_ce_examples(qrels_df):
    examples = []
    for _, row in qrels_df.iterrows():
        q = q_map.get(row["query_id"])
        d = doc_map.get(row["document_id"])
        if q and d:
            examples.append(InputExample(texts=[q, d], label=float(row["relevance"]) / 3.0))
    return examples

def add_hard_negs(examples, qids, cands_dict, top_k=30):
    enriched = list(examples)
    n = 0
    for qid in qids:
        if qid not in q_map or qid not in cands_dict: continue
        judged = set(qrels_g.get(qid, {}).keys())
        for did in cands_dict[qid][:top_k]:
            if did not in judged and did in doc_map:
                enriched.append(InputExample(texts=[q_map[qid], doc_map[did]], label=0.0))
                n += 1
    return enriched, n

def train_ce(examples, epochs=2, warmup=100, batch_size=8, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    ce = CrossEncoder("cross-encoder/ettin-reranker-68m-v1", num_labels=1, device=device)
    ce.fit(train_dataloader=DataLoader(examples, shuffle=True, batch_size=batch_size, num_workers=0),
           epochs=epochs, warmup_steps=warmup, show_progress_bar=True)
    return ce

def ce_ensemble_scores(ces, query, cand_ids):
    """Average scores from multiple CEs for one query."""
    pairs = [[query, doc_map[d]] for d in cand_ids]
    avg = np.zeros(len(cand_ids))
    for ce in ces:
        avg += np.asarray(ce.predict(pairs, batch_size=64, show_progress_bar=False))
    avg /= len(ces)
    return dict(zip(cand_ids, avg.tolist()))

print("CE helpers defined.")

In [ ]:
# Cell 6: Train 10-fold CE ensemble on ALL data
print(f"Training {N_FOLDS}-fold CE ensemble on ALL training data...")
all_train_qids = set(train_q["query_id"])
final_ces = []

for i in range(N_FOLDS):
    fold_hold = all_folds[i]
    fold_train = all_train_qids - fold_hold
    fold_qrels_df = qrels[qrels["query_id"].isin(fold_train)]
    fold_ex = build_ce_examples(fold_qrels_df)
    fold_enr, n = add_hard_negs(fold_ex, fold_train, train_cands, top_k=30)
    print(f"  Fold {i}: {len(fold_train)} queries, {len(fold_enr)} examples ({n} HN)")
    final_ces.append(train_ce(fold_enr, seed=42+i))

print(f"\nTrained {len(final_ces)} CEs.")

In [ ]:
# Cell 7: CE ensemble scores for val + test
print("Scoring val candidates with 10-CE ensemble...")
val_ce_scores = {}
for qid in sorted(val_qids):
    if qid not in val_cands: continue
    val_ce_scores[qid] = ce_ensemble_scores(final_ces, q_map[qid], val_cands[qid])

# Val nDCG with CE-only
val_ndcgs_ce = []
for qid in sorted(val_qids):
    if qid not in val_ce_scores or qid not in qrels_g: continue
    ranked = sorted(val_ce_scores[qid].keys(), key=lambda d: -val_ce_scores[qid][d])
    val_ndcgs_ce.append(ndcg_at_k(ranked[:5], qrels_g[qid]))
ce_only_val = float(np.mean(val_ndcgs_ce))
print(f"CE-only val nDCG@5: {ce_only_val:.4f} ({len(val_ndcgs_ce)} queries)")

print("\nScoring test candidates with 10-CE ensemble...")
test_ce_scores = {}
for qid in test_q["query_id"]:
    test_ce_scores[qid] = ce_ensemble_scores(final_ces, tq_map[qid], test_cands[qid])

print(f"Val: {len(val_ce_scores)} queries | Test: {len(test_ce_scores)} queries")

# Free CE models from GPU memory
del final_ces; torch.cuda.empty_cache()
print("CE scoring done, models freed.")

In [ ]:
# Cell 8: Pretrained reranker (zero-shot) for val + test
import time

print("Loading pretrained reranker: BAAI/bge-reranker-v2-m3...")
pretrained_ce = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)
print("Pretrained reranker loaded.")

def pretrained_scores(query, cand_ids):
    """Score candidates with pretrained reranker."""
    pairs = [[query, doc_map[d]] for d in cand_ids]
    scores = pretrained_ce.predict(pairs, batch_size=64, show_progress_bar=False)
    return dict(zip(cand_ids, scores.tolist()))

print(f"Scoring val candidates with pretrained reranker...")
val_pt_scores = {}
for i, qid in enumerate(sorted(val_qids)):
    if qid not in val_cands: continue
    val_pt_scores[qid] = pretrained_scores(q_map[qid], val_cands[qid])
    if (i + 1) % 20 == 0:
        print(f"  Val: {i+1}/{len(val_ce_scores)}")

# Val nDCG with pretrained-only
val_ndcgs_pt = []
for qid in sorted(val_qids):
    if qid not in val_pt_scores or qid not in qrels_g: continue
    ranked = sorted(val_pt_scores[qid].keys(), key=lambda d: -val_pt_scores[qid][d])
    val_ndcgs_pt.append(ndcg_at_k(ranked[:5], qrels_g[qid]))
pt_only_val = float(np.mean(val_ndcgs_pt))
print(f"Pretrained-only val nDCG@5: {pt_only_val:.4f}")

print(f"\nScoring test candidates with pretrained reranker...")
test_pt_scores = {}
for i, qid in enumerate(test_q["query_id"]):
    test_pt_scores[qid] = pretrained_scores(tq_map[qid], test_cands[qid])
    if (i + 1) % 50 == 0:
        print(f"  Test: {i+1}/{len(test_ce_scores)}")

print(f"Pretrained scoring done. Val: {len(val_pt_scores)} | Test: {len(test_pt_scores)}")

del pretrained_ce; torch.cuda.empty_cache()

In [ ]:
# Cell 9: Tune alpha + evaluate + produce submission
import numpy as np

print("=" * 70)
print("v15 RESULTS")
print("=" * 70)

print(f"\n  10-fold CE-only val nDCG@5: {ce_only_val:.4f}")
print(f"  Pretrained-only val nDCG@5: {pt_only_val:.4f}")

def min_max_normalize(scores_dict):
    """Normalize scores to [0, 1] range."""
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx == mn:
        return {k: 0.5 for k in scores_dict}
    return {k: (v - mn) / (mx - mn) for k, v in scores_dict.items()}

# Tune alpha on val: score = alpha * CE_norm + (1-alpha) * Pretrained_norm
best_alpha = 0.5
best_blend_val = 0.0

for alpha in np.arange(0.0, 1.05, 0.05):
    blend_ndcgs = []
    for qid in sorted(val_qids):
        if qid not in val_ce_scores or qid not in val_pt_scores or qid not in qrels_g:
            continue
        ce_norm = min_max_normalize(val_ce_scores[qid])
        pt_norm = min_max_normalize(val_pt_scores[qid])
        all_docs = set(ce_norm.keys()) | set(pt_norm.keys())
        blended = {d: alpha * ce_norm.get(d, 0) + (1 - alpha) * pt_norm.get(d, 0) for d in all_docs}
        ranked = sorted(blended.keys(), key=lambda d: -blended[d])
        blend_ndcgs.append(ndcg_at_k(ranked[:5], qrels_g[qid]))
    mean_ndcg = float(np.mean(blend_ndcgs))
    if mean_ndcg > best_blend_val:
        best_blend_val = mean_ndcg
        best_alpha = alpha

print(f"\n  Best blend: alpha={best_alpha:.2f} → val nDCG@5: {best_blend_val:.4f}")
print(f"  Improvement over CE-only: {best_blend_val - ce_only_val:+.4f}")

# Show alpha sweep
print(f"\n  Alpha sweep (CE weight):")
for alpha in np.arange(0.0, 1.05, 0.1):
    blend_ndcgs = []
    for qid in sorted(val_qids):
        if qid not in val_ce_scores or qid not in val_pt_scores or qid not in qrels_g:
            continue
        ce_norm = min_max_normalize(val_ce_scores[qid])
        pt_norm = min_max_normalize(val_pt_scores[qid])
        all_docs = set(ce_norm.keys()) | set(pt_norm.keys())
        blended = {d: alpha * ce_norm.get(d, 0) + (1 - alpha) * pt_norm.get(d, 0) for d in all_docs}
        ranked = sorted(blended.keys(), key=lambda d: -blended[d])
        blend_ndcgs.append(ndcg_at_k(ranked[:5], qrels_g[qid]))
    print(f"    alpha={alpha:.1f}: {np.mean(blend_ndcgs):.4f}")

# Also try RRF fusion
val_ndcgs_rrf = []
for qid in sorted(val_qids):
    if qid not in val_ce_scores or qid not in val_pt_scores or qid not in qrels_g:
        continue
    ce_ranked = sorted(val_ce_scores[qid].keys(), key=lambda d: -val_ce_scores[qid][d])
    pt_ranked = sorted(val_pt_scores[qid].keys(), key=lambda d: -val_pt_scores[qid][d])
    fused = rrf_fuse([ce_ranked, pt_ranked], k=60)
    val_ndcgs_rrf.append(ndcg_at_k(fused[:5], qrels_g[qid]))
rrf_val = float(np.mean(val_ndcgs_rrf))
print(f"\n  RRF(CE+Pretrained) val nDCG@5: {rrf_val:.4f} ({rrf_val - ce_only_val:+.4f})")

# Pick best method
methods = {"CE-only": ce_only_val, f"Blend(a={best_alpha:.2f})": best_blend_val, "RRF": rrf_val}
best_method = max(methods, key=methods.get)
print(f"\n  BEST METHOD: {best_method} = {methods[best_method]:.4f}")
print("=" * 70)

# Produce submission with best method
rows = []
for qid in test_q["query_id"]:
    if best_method.startswith("Blend"):
        ce_norm = min_max_normalize(test_ce_scores[qid])
        pt_norm = min_max_normalize(test_pt_scores[qid])
        all_docs = set(ce_norm.keys()) | set(pt_norm.keys())
        blended = {d: best_alpha * ce_norm.get(d, 0) + (1 - best_alpha) * pt_norm.get(d, 0) for d in all_docs}
        ranked = sorted(blended.keys(), key=lambda d: -blended[d])
    elif best_method == "RRF":
        ce_ranked = sorted(test_ce_scores[qid].keys(), key=lambda d: -test_ce_scores[qid][d])
        pt_ranked = sorted(test_pt_scores[qid].keys(), key=lambda d: -test_pt_scores[qid][d])
        ranked = rrf_fuse([ce_ranked, pt_ranked], k=60)
    else:
        ranked = sorted(test_ce_scores[qid].keys(), key=lambda d: -test_ce_scores[qid][d])
    for did in ranked[:5]:
        rows.append({"QueryId": qid, "DocumentId": did})

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv ({best_method}): {len(sub)} rows, {sub['QueryId'].nunique()} queries")

# Also produce submissions for each method for manual comparison
# Blend with best alpha
rows_blend = []
for qid in test_q["query_id"]:
    ce_norm = min_max_normalize(test_ce_scores[qid])
    pt_norm = min_max_normalize(test_pt_scores[qid])
    all_docs = set(ce_norm.keys()) | set(pt_norm.keys())
    blended = {d: best_alpha * ce_norm.get(d, 0) + (1 - best_alpha) * pt_norm.get(d, 0) for d in all_docs}
    ranked = sorted(blended.keys(), key=lambda d: -blended[d])
    for did in ranked[:5]:
        rows_blend.append({"QueryId": qid, "DocumentId": did})
pd.DataFrame(rows_blend).to_csv("submission_blend.csv", index=False)

# CE-only backup
rows_ce = []
for qid in test_q["query_id"]:
    ranked = sorted(test_ce_scores[qid].keys(), key=lambda d: -test_ce_scores[qid][d])
    for did in ranked[:5]:
        rows_ce.append({"QueryId": qid, "DocumentId": did})
pd.DataFrame(rows_ce).to_csv("submission_ce_only.csv", index=False)

print(f"submission_blend.csv: {len(rows_blend)} rows")
print(f"submission_ce_only.csv: {len(rows_ce)} rows")
print("\nDone!")